### Import

In [1]:
from rdflib import Graph, Literal
from rdflib.namespace import RDF, SKOS
from natsort import index_natsorted
import pandas as pd

### Parse the vocabulary

In [2]:
g = Graph()
g.parse("openatlas-bone.ttl", format="turtle")
triples = list(g.triples((None, None, None)))
print(len(triples))

18503


### Replace skos:notation with skos:altLabel

In [3]:
triples = list(g.triples((None, SKOS.notation, None)))
print(len(triples))
for triple in triples[0:5]:
    print(triple)

2173
(rdflib.term.URIRef('https://vocabs.acdh.oeaw.ac.at/openatlas-bone/C02-4'), rdflib.term.URIRef('http://www.w3.org/2004/02/skos/core#notation'), rdflib.term.Literal('A02.2.02.204'))
(rdflib.term.URIRef('https://vocabs.acdh.oeaw.ac.at/openatlas-bone/C02-4'), rdflib.term.URIRef('http://www.w3.org/2004/02/skos/core#notation'), rdflib.term.Literal('C02'))
(rdflib.term.URIRef('https://vocabs.acdh.oeaw.ac.at/openatlas-bone/C02-5'), rdflib.term.URIRef('http://www.w3.org/2004/02/skos/core#notation'), rdflib.term.Literal('C02'))
(rdflib.term.URIRef('https://vocabs.acdh.oeaw.ac.at/openatlas-bone/C02-11'), rdflib.term.URIRef('http://www.w3.org/2004/02/skos/core#notation'), rdflib.term.Literal('C02'))
(rdflib.term.URIRef('https://vocabs.acdh.oeaw.ac.at/openatlas-bone/C02-12'), rdflib.term.URIRef('http://www.w3.org/2004/02/skos/core#notation'), rdflib.term.Literal('C02'))


In [4]:
for subject, _, obj in triples:
    # Remove the original triple
    g.remove((subject, SKOS.notation, obj))

    # Create a new literal with the same lexical value but language tag zxx
    alt_label = Literal(str(obj), lang="zxx")

    # Add the new triple
    g.add((subject, SKOS.altLabel, alt_label))

In [5]:
triples = list(g.triples((None, SKOS.altLabel, None)))
print(len(triples))
for triple in triples[0:5]:
    print(triple)

2173
(rdflib.term.URIRef('https://vocabs.acdh.oeaw.ac.at/openatlas-bone/C02-4'), rdflib.term.URIRef('http://www.w3.org/2004/02/skos/core#altLabel'), rdflib.term.Literal('A02.2.02.204', lang='zxx'))
(rdflib.term.URIRef('https://vocabs.acdh.oeaw.ac.at/openatlas-bone/C02-4'), rdflib.term.URIRef('http://www.w3.org/2004/02/skos/core#altLabel'), rdflib.term.Literal('C02', lang='zxx'))
(rdflib.term.URIRef('https://vocabs.acdh.oeaw.ac.at/openatlas-bone/C02-5'), rdflib.term.URIRef('http://www.w3.org/2004/02/skos/core#altLabel'), rdflib.term.Literal('C02', lang='zxx'))
(rdflib.term.URIRef('https://vocabs.acdh.oeaw.ac.at/openatlas-bone/C02-11'), rdflib.term.URIRef('http://www.w3.org/2004/02/skos/core#altLabel'), rdflib.term.Literal('C02', lang='zxx'))
(rdflib.term.URIRef('https://vocabs.acdh.oeaw.ac.at/openatlas-bone/C02-12'), rdflib.term.URIRef('http://www.w3.org/2004/02/skos/core#altLabel'), rdflib.term.Literal('C02', lang='zxx'))


### Export the vocabulary

In [6]:
g.serialize(destination="openatlas-bone.ttl")

<Graph identifier=Nee4710c7dd2549f69460b75b255b6250 (<class 'rdflib.graph.Graph'>)>

In [7]:
g = Graph()
g.parse("openatlas-bone.ttl", format="turtle")
triples = list(g.triples((None, None, None)))
print(len(triples))

18503


### Create table

In [8]:
def get_label(graph, subject, lang):
    for label in graph.objects(subject, SKOS.prefLabel):
        if label.language == lang:
            return label
    return None

In [9]:
concepts = []

for s, p, o in g.triples((None, RDF.type, SKOS.Concept)):
    concept_dict = {
        "uri": s.rsplit("/", 1)[-1],
        "link": "https://vocabs.acdh-dev.oeaw.ac.at/openatlas-bone_browse/en/page/" + s.rsplit("/", 1)[-1],
        "label_en": get_label(g, s, "en"),
        "label_de": get_label(g, s, "de"),
        "label_la": get_label(g, s, "la"),
        "altLabels": sorted([o.value for _, _, o in g.triples((s, SKOS.altLabel, None))]),
        "broader": sorted([o.rsplit("/", 1)[-1] for _, _, o in g.triples((s, SKOS.broader, None))]),
    }
    concepts.append(concept_dict)

df = pd.DataFrame(concepts)

df["broader"] = df["broader"].apply(
    lambda x: x[0] if isinstance(x, list) and len(x) == 1 else x
)

df = df.iloc[index_natsorted(df["uri"])].reset_index(drop=True)
df.to_csv("concepts.csv", index=False)

In [10]:
duplicates_en = (
    df[df.duplicated(subset="label_en", keep=False)]
    .sort_values("label_en")
)
duplicates_en.to_csv("duplicates_en.csv", index=False)

In [11]:
duplicates_de = (
    df[df.duplicated(subset="label_de", keep=False)]
    .sort_values("label_de")
)
duplicates_de.to_csv("duplicates_de.csv", index=False)

In [12]:
duplicates_la = (
    df[df.duplicated(subset="label_la", keep=False)]
    .sort_values("label_la")
)
duplicates_la.to_csv("duplicates_la.csv", index=False)

In [13]:
common = (
    duplicates_en.merge(duplicates_de, on="uri", suffixes=("_en", "_de"))
       .merge(duplicates_la, on="uri")
)